# Instalación de dependencias y fijación de semillas

In [1]:
!pip install -q -U transformers==4.46.2 accelerate==1.1.1 peft==0.13.2 \
    datasets==3.1.0 scikit-learn==1.5.2 chromadb sentence-transformers rank-bm25 pillow pandas

import os, re, json, time, glob, random, collections
from pathlib import Path
from typing import Any, Dict, List, Optional
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageEnhance, ImageOps

SEED = 42
def fijar_semillas(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semillas(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo activo: {DEVICE}")
assert DEVICE == "cuda", "Activa GPU T4 en Colab (Entorno de ejecución > Cambiar tipo de entorno)."

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
diffusers 0.42.1 req

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Dispositivo activo: cuda


# Montar Google Drive, clonar repositorio y ubicar rutas de M1 y M3

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    EN_COLAB = True
except Exception:
    EN_COLAB = False

BASE_DRIVE = "/content/drive/MyDrive/personal-shopper-ia" if EN_COLAB else "."
ADAPTER_PATH = f"{BASE_DRIVE}/qwen25-1.5b-style-extractor-lora/adapter_final"

# Clonar repo si no está presente en el entorno actual
REPO_DIR = "Isabela_Mendoza_Juan_Manuel_Lopera_Cesar_Montoya"
if not os.path.exists("notebooks/M3") and not os.path.exists(REPO_DIR):
    !git clone -b cesar https://github.com/CesarMontoyag1/Isabela_Mendoza_Juan_Manuel_Lopera_Cesar_Montoya.git

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)

M3_DIR = "notebooks/M3" if os.path.exists("notebooks/M3/corpus/corpus_look_es.csv") else "."
M4_DIR = "notebooks/M4"
os.makedirs(f"{M4_DIR}/eval", exist_ok=True)
os.makedirs(f"{M4_DIR}/resultados", exist_ok=True)
os.makedirs(f"{M4_DIR}/imagenes_test", exist_ok=True)

LOOK_CSV      = os.path.join(M3_DIR, "corpus", "corpus_look_es.csv")
LABEL_CSV     = os.path.join(M3_DIR, "corpus", "corpus_label_es.csv")
EVAL_M3_PATH  = os.path.join(M3_DIR, "eval", "eval_set_rag.json")
CHROMA_DIR    = os.path.join(M4_DIR, "chroma_db_m4")

print("Adaptador LoRA M1 existe :", os.path.exists(ADAPTER_PATH), "->", ADAPTER_PATH)
print("Corpus M3 existe         :", os.path.exists(LOOK_CSV), "->", LOOK_CSV)
print("Eval set M3 existe       :", os.path.exists(EVAL_M3_PATH), "->", EVAL_M3_PATH)

Mounted at /content/drive
Cloning into 'Isabela_Mendoza_Juan_Manuel_Lopera_Cesar_Montoya'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (317/317), done.
remote: Compressing objects: 100% (240/240), done.
remote: Total 317 (delta 141), reused 195 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (317/317), 1.19 MiB | 10.01 MiB/s, done.
Resolving deltas: 100% (141/141), done.
Adaptador LoRA M1 existe : True -> /content/drive/MyDrive/personal-shopper-ia/qwen25-1.5b-style-extractor-lora/adapter_final
Corpus M3 existe         : True -> notebooks/M3/corpus/corpus_look_es.csv
Eval set M3 existe       : True -> notebooks/M3/eval/eval_set_rag.json


# Carga del Corpus e Índice Híbrido (ChromaDB + BM25) de M3

In [3]:
import csv
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

EMBED_MODEL = "intfloat/multilingual-e5-small"
TOP_K = 5

def _norm(v):
    mapa = {"Business": "Trabajo"}
    return "/".join(mapa.get(p.strip(), p.strip()) for p in str(v).split("/"))

def leer_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

looks = leer_csv(LOOK_CSV)
labels = leer_csv(LABEL_CSV)
labels_by_id = {row["itemID"]: row for row in labels}

def genero_outfit(items):
    gens = [labels_by_id[i]["gender"] for i in items if i in labels_by_id]
    if not gens:
        return "Unisex"
    for g in ("De mujer", "De hombre"):
        if g in gens:
            return g
    return "Unisex"

def construir_documento(outfit):
    items = [i.strip() for i in outfit["items"].split(",") if i.strip()]
    partes = [f"Outfit {outfit['outfitID']}: {outfit['look']}"]
    partes.append(f"Estación: {_norm(outfit['season'])}. Ocasión: {_norm(outfit['occasion'])}.")
    detalles = []
    for i in items:
        p = labels_by_id.get(i)
        if not p:
            continue
        d = f"- {p['title']} ({p['category']}): {p['color']}, {p['materials']}, {p['outline']}"
        if p.get("detail"):
            d += f". {p['detail']}"
        detalles.append(d)
    if detalles:
        partes.append("Prendas:")
        partes.extend(detalles)
    return "\n".join(partes)

documentos, metadatos, doc_ids = [], [], []
for o in looks:
    items = [i.strip() for i in o["items"].split(",") if i.strip()]
    documentos.append(construir_documento(o))
    doc_ids.append(o["outfitID"])
    metadatos.append({
        "outfitID": o["outfitID"],
        "season": _norm(o["season"]),
        "occasion": _norm(o["occasion"]),
        "gender": genero_outfit(items),
        "coleccion": o.get("coleccion", ""),
    })

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

def embed_passages(textos):
    return embedder.encode(["passage: " + t for t in textos], normalize_embeddings=True, show_progress_bar=False).tolist()

def embed_queries(textos):
    return embedder.encode(["query: " + t for t in textos], normalize_embeddings=True, show_progress_bar=False).tolist()

client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    client.delete_collection("outfits_m4")
except Exception:
    pass
col = client.create_collection("outfits_m4", metadata={"hnsw:space": "cosine"})

for i in range(0, len(documentos), 128):
    sl = slice(i, i + 128)
    col.add(ids=doc_ids[sl], documents=documentos[sl], embeddings=embed_passages(documentos[sl]), metadatas=metadatos[sl])

def tokenizar(t):
    return re.findall(r"\w+", t.lower())

bm25 = BM25Okapi([tokenizar(d) for d in documentos])
print(f"Índice híbrido listo: {col.count()} outfits en Chroma + BM25.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Índice híbrido listo: 800 outfits en Chroma + BM25.


# Carga Unificada de Modelos (M1 LoRA + M3 Base + M4 Visual VLM)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

GEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
VIS_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

print("1/2 Cargando Qwen2.5-1.5B-Instruct + Adaptador LoRA M1...")
gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
if gen_tok.pad_token is None:
    gen_tok.pad_token = gen_tok.eos_token

base_gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID,
    torch_dtype=torch.float16,
    device_map=None,
).to(DEVICE)

if os.path.exists(ADAPTER_PATH):
    gen_model = PeftModel.from_pretrained(base_gen_model, ADAPTER_PATH).to(DEVICE)
    TIENE_LORA = True
    print("   -> Adaptador LoRA de M1 cargado correctamente.")
else:
    gen_model = base_gen_model
    TIENE_LORA = False
    print("   -> AVISO: No se halló adapter_final en Drive; usando modelo base.")

gen_model.eval()

print("2/2 Cargando modelo visual Qwen2-VL-2B-Instruct (M4)...")
vis_processor = AutoProcessor.from_pretrained(
    VIS_MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)
vis_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VIS_MODEL_ID,
    torch_dtype=torch.float16,
    device_map=None,
).to(DEVICE)
vis_model.eval()

vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
print(f"Modelos cargados. Memoria VRAM máxima ocupada hasta ahora: {vram_gb:.2f} GB / 15.00 GB")

1/2 Cargando Qwen2.5-1.5B-Instruct + Adaptador LoRA M1...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

... (13 outputs left out)

`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Modelos cargados. Memoria VRAM máxima ocupada hasta ahora: 7.46 GB / 15.00 GB


# Componente Visual M4 (Percepción + Válvula Visual Anti-Alucinación)

In [5]:
VISUAL_SYSTEM_PROMPT = (
    "Eres el módulo de visión de un personal shopper de moda. Analiza la imagen proporcionada y "
    "responde ÚNICAMENTE con un objeto JSON válido con esta estructura exacta:\n"
    "{\n"
    '  "es_moda": true o false,\n'
    '  "genero_aparente": "De mujer" | "De hombre" | "Unisex" | null,\n'
    '  "clima_sugerido": "frio" | "calido" | "templado" | null,\n'
    '  "descripcion": "descripción breve en español de las prendas, colores, corte/fit y estilo visibles"\n'
    "}\n\n"
    "REGLA ESTRICTA: Si la imagen NO muestra ropa, prendas de vestir, calzado, accesorios de moda "
    "o un outfit de una persona (por ejemplo, si muestra autos, comida, animales, paisajes, gráficos, "
    'documentos o ruido), devuelve {"es_moda": false, "genero_aparente": null, "clima_sugerido": null, '
    '"descripcion": "OUT_OF_DOMAIN"}. No inventes prendas.'
)

def parse_json_seguro(texto: str) -> Dict[str, Any]:
    if not texto:
        return {}
    ini, fin = texto.find("{"), texto.rfind("}")
    if ini == -1 or fin == -1 or fin < ini:
        return {}
    try:
        obj = json.loads(texto[ini:fin + 1])
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}

@torch.no_grad()
def analizar_imagen_m4(imagen: Image.Image) -> Dict[str, Any]:
    """Procesa una imagen PIL y extrae atributos de moda o detecta fuera de dominio."""
    if imagen.mode != "RGB":
        imagen = imagen.convert("RGB")

    mensajes = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": VISUAL_SYSTEM_PROMPT},
            ],
        }
    ]
    prompt_txt = vis_processor.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    inputs = vis_processor(text=[prompt_txt], images=[imagen], padding=True, return_tensors="pt").to(DEVICE)

    out = vis_model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=vis_processor.tokenizer.eos_token_id,
    )
    gen_ids = out[:, inputs["input_ids"].shape[1]:]
    respuesta_cruda = vis_processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

    parsed = parse_json_seguro(respuesta_cruda)
    if not parsed:
        # Fallback conservador si el VLM respondió texto libre
        es_ood = "out_of_domain" in respuesta_cruda.lower() or "no muestra ropa" in respuesta_cruda.lower()
        return {
            "es_moda": not es_ood,
            "genero_aparente": None,
            "clima_sugerido": None,
            "descripcion": "OUT_OF_DOMAIN" if es_ood else respuesta_cruda[:200],
        }

    return {
        "es_moda": bool(parsed.get("es_moda", False)),
        "genero_aparente": parsed.get("genero_aparente"),
        "clima_sugerido": parsed.get("clima_sugerido"),
        "descripcion": str(parsed.get("descripcion", "")).strip(),
    }

# Integración de M1 (Extractor LoRA) + M3 (RAG Híbrido + HyDE + Tool) + M4

In [6]:
# --- Ontología y Prompt de M1 ---
ONTOLOGIA_M1 = {
    "estilo": ["Casual", "Formal", "Minimalista", "Urbano", "Bohemio", "Deportivo", "Clasico"],
    "ocasion": ["boda", "trabajo", "fin_de_semana", "viaje", "deporte", "evento_formal"],
    "clima": ["calido", "frio", "templado"],
    "paleta": ["neutros", "pasteles", "oscuros", "colores_vivos", "monocromatico"],
    "fit": ["holgado", "regular", "ajustado", "oversized"],
}

SYSTEM_PROMPT_M1 = (
    "Eres un asistente que extrae el perfil de estilo de un cliente a partir de su mensaje "
    "de chat en una tienda de moda online.\n"
    "Debes devolver UNICAMENTE un objeto JSON con los campos que el cliente haya mencionado "
    "o se puedan inferir claramente, usando SOLO estos campos y valores posibles:\n\n"
    "- estilo: Casual, Formal, Minimalista, Urbano, Bohemio, Deportivo, Clasico\n"
    "- ocasion: boda, trabajo, fin_de_semana, viaje, deporte, evento_formal\n"
    "- clima: calido, frio, templado\n"
    "- paleta: neutros, pasteles, oscuros, colores_vivos, monocromatico\n"
    "- fit: holgado, regular, ajustado, oversized\n\n"
    "No incluyas campos que no se puedan inferir del mensaje. No agregues texto fuera del "
    "JSON. No inventes valores fuera de esta lista."
)

@torch.no_grad()
def extraer_perfil_m1(texto_usuario: str) -> Dict[str, str]:
    """Ejecuta el extractor fine-tuned de M1 (con LoRA activo)."""
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT_M1},
        {"role": "user", "content": texto_usuario},
    ]
    prompt = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tok(prompt, return_tensors="pt").to(DEVICE)
    out = gen_model.generate(**inputs, max_new_tokens=96, do_sample=False, pad_token_id=gen_tok.eos_token_id)
    raw = gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    obj = parse_json_seguro(raw)
    return {k: v for k, v in obj.items() if k in ONTOLOGIA_M1 and v in ONTOLOGIA_M1[k]}

@torch.no_grad()
def generar_base(system_prompt: str, user_prompt: str, max_new_tokens: int = 320) -> str:
    """Genera texto con el modelo base (desactivando temporalmente el adaptador LoRA)."""
    msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tok(prompt, return_tensors="pt").to(DEVICE)

    if TIENE_LORA:
        with gen_model.disable_adapter():
            out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=gen_tok.eos_token_id)
    else:
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=gen_tok.eos_token_id)

    return gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# --- Prompts y Recuperación de M3 ---
SYSTEM_PROMPT_M3 = (
    "Eres un personal shopper de un e-commerce de moda. Respondes en español, breve y concreto.\n"
    "REGLA 1 (recomendar): si el CONTEXTO contiene outfits de ropa, SIEMPRE recomienda el o los "
    "que mejor encajen con la petición y cítalos por su identificador (por ejemplo, B_b12). "
    "El contexto son outfits de ropa; úsalos aunque no sean una coincidencia perfecta.\n"
    'REGLA 2 (válvula de escape): usa la frase "No encuentro esa información en el catálogo" '
    "ÚNICAMENTE cuando la pregunta NO es sobre recomendar ropa, sino sobre precios, envíos, "
    "tallas numéricas (38, 40...) o políticas de la tienda (devoluciones). En esos casos no hay "
    "outfit que recomendar.\n"
    "NUNCA uses la válvula si la pregunta pide ropa y el contexto trae outfits. No inventes prendas "
    "que no estén en el contexto."
)
VALVULA = "No encuentro esa información en el catálogo"

HYDE_SYS = (
    "Eres un redactor de catálogo de moda. Dada una petición de un cliente, redacta en "
    "2-3 frases un outfit hipotético que la satisfaría (estilo, prendas, colores, estación, "
    "ocasión). Solo la descripción, en español."
)

SINONIMOS_GENERO = {
    "De hombre": ["hombre", "masculino", "caballero", "hombres", "para él", "para el"],
    "De mujer": ["mujer", "femenino", "dama", "mujeres", "para ella", "chica", "vestido", "falda"],
}
MAPA_CLIMA_M1_A_M3 = {"frio": "Invierno", "calido": "Verano", "templado": "Primavera"}
MAPA_OCASION_M1_A_M3 = {
    "boda": "Eventos sociales",
    "evento_formal": "Eventos sociales",
    "trabajo": "Trabajo",
    "viaje": "Viaje",
    "deporte": "Deporte",
}

def construir_filtro_integrado(consulta: str, perfil_m1: Dict[str, str], info_visual: Optional[Dict[str, Any]]) -> Dict[str, str]:
    """Une las señales de M4 (visión) + M1 (JSON LoRA) + palabras clave para la herramienta de M3."""
    args: Dict[str, str] = {}
    q = consulta.lower()

    # 1. Género (texto explícito o inferido por visión M4)
    for canonico, claves in SINONIMOS_GENERO.items():
        if any(k in q for k in claves):
            args["gender"] = canonico
            break
    if "gender" not in args and info_visual and info_visual.get("genero_aparente") in ("De mujer", "De hombre"):
        args["gender"] = info_visual["genero_aparente"]

    # 2. Estación desde M1 o M4
    if "clima" in perfil_m1 and perfil_m1["clima"] in MAPA_CLIMA_M1_A_M3:
        args["season"] = MAPA_CLIMA_M1_A_M3[perfil_m1["clima"]]
    elif info_visual and info_visual.get("clima_sugerido") in MAPA_CLIMA_M1_A_M3:
        args["season"] = MAPA_CLIMA_M1_A_M3[info_visual["clima_sugerido"]]

    # 3. Ocasión desde M1
    if "ocasion" in perfil_m1 and perfil_m1["ocasion"] in MAPA_OCASION_M1_A_M3:
        args["occasion"] = MAPA_OCASION_M1_A_M3[perfil_m1["ocasion"]]

    return args

def buscar_denso(query_txt: str, k: int = TOP_K):
    res = col.query(query_embeddings=embed_queries([query_txt]), n_results=k, include=["documents", "metadatas"])
    return list(zip([m["outfitID"] for m in res["metadatas"][0]], res["documents"][0]))

def buscar_bm25(query_txt: str, k: int = 20, candidatos_idx=None):
    scores = bm25.get_scores(tokenizar(query_txt))
    idx = range(len(documentos)) if candidatos_idx is None else candidatos_idx
    ranked = sorted(idx, key=lambda i: scores[i], reverse=True)[:k]
    return [(doc_ids[i], documentos[i]) for i in ranked]

def rrf(listas_ranking, k_rrf: int = 60, top: int = TOP_K):
    puntajes = collections.defaultdict(float)
    docs = {}
    for lista in listas_ranking:
        for rank, (i, d) in enumerate(lista):
            puntajes[i] += 1.0 / (k_rrf + rank)
            docs[i] = d
    orden = sorted(puntajes, key=puntajes.get, reverse=True)[:top]
    return [(i, docs[i]) for i in orden]

def armar_prompt_m3(pregunta: str, contextos: List[str], perfil_m1: Dict[str, str]) -> str:
    ctx = "\n\n".join(contextos) if contextos else "(sin resultados)"
    perfil_str = json.dumps(perfil_m1, ensure_ascii=False) if perfil_m1 else "{}"
    return (
        f"PERFIL DE ESTILO DETECTADO: {perfil_str}\n\n"
        f"CONTEXTO (outfits del catálogo):\n{ctx}\n\n"
        f"PREGUNTA DEL CLIENTE: {pregunta}\n\n"
        "El contexto SÍ trae outfits: recomiéndale al cliente el o los que mejor encajen y "
        "cita sus identificadores. Explica en 1-2 frases por qué encaja. Solo usa la válvula de escape "
        "si la pregunta es de precio, envío, talla numérica o políticas de la tienda."
    )

def sistema_multimodal(pregunta: str, imagen: Optional[Image.Image] = None) -> Dict[str, Any]:
    """Sistema integrador M1 + M3 + M4."""
    t0 = time.time()
    info_visual = None
    consulta_enriquecida = pregunta

    # --- PASO 1: Módulo Visual (M4) ---
    if imagen is not None:
        info_visual = analizar_imagen_m4(imagen)
        if not info_visual["es_moda"]:
            # Válvula de escape visual ante imágenes fuera de dominio (adversariales)
            return {
                "respuesta": f"{VALVULA}: la imagen adjunta no corresponde a prendas de vestir o moda.",
                "contextos": [],
                "outfits": [],
                "latencia_s": time.time() - t0,
                "uso_vision": True,
                "uso_herramienta": False,
                "valvula_activada": True,
                "perfil_m1": {},
                "info_visual": info_visual,
                "filtro": {},
            }
        consulta_enriquecida = f"{pregunta}. Prenda o referencia visual adjunta: {info_visual['descripcion']}"

    # --- PASO 2: Extractor de Perfil con LoRA (M1) ---
    perfil_m1 = extraer_perfil_m1(consulta_enriquecida)

    # --- PASO 3: Herramienta de Filtrado + Recuperación Híbrida & HyDE (M3) ---
    args = construir_filtro_integrado(consulta_enriquecida, perfil_m1, info_visual)
    uso_herramienta = bool(args)

    if args:
        idx_validos = [k for k, m in enumerate(metadatos) if all(v in m.get(c, "") for c, v in args.items())]
    else:
        idx_validos = list(range(len(documentos)))
    if not idx_validos:
        idx_validos = list(range(len(documentos)))

    pool = 20
    sub_ids = set(doc_ids[i] for i in idx_validos)
    densos = [(i, d) for i, d in buscar_denso(consulta_enriquecida, k=pool * 3) if i in sub_ids][:pool]
    lexicos = buscar_bm25(consulta_enriquecida, k=pool, candidatos_idx=idx_validos)

    hip = generar_base(HYDE_SYS, f"Petición: {consulta_enriquecida}\nDescripción del outfit ideal:", max_new_tokens=100)
    hyde_densos = [(i, d) for i, d in buscar_denso(hip, k=pool * 3) if i in sub_ids][:pool]

    finalistas = rrf([densos, lexicos, hyde_densos], top=TOP_K) if (densos or lexicos or hyde_densos) else buscar_denso(consulta_enriquecida, k=TOP_K)
    contextos = [d for _, d in finalistas]

    # --- PASO 4: Generación Final ---
    respuesta = generar_base(SYSTEM_PROMPT_M3, armar_prompt_m3(consulta_enriquecida, contextos, perfil_m1))
    valvula_act = VALVULA.lower() in respuesta.lower()

    return {
        "respuesta": respuesta,
        "contextos": contextos,
        "outfits": [i for i, _ in finalistas],
        "latencia_s": time.time() - t0,
        "uso_vision": imagen is not None,
        "uso_herramienta": uso_herramienta,
        "valvula_activada": valvula_act,
        "perfil_m1": perfil_m1,
        "info_visual": info_visual,
        "filtro": args,
    }

# Construcción del Dataset Multimodal (eval_set_m4.json) y Casos de Prueba

In [7]:
from datasets import load_dataset

with open(EVAL_M3_PATH, encoding="utf-8") as f:
    eval_texto_m3 = json.load(f)

# Obtener 4 imágenes reales de ropa desde Hugging Face
imagenes_ropa = []
try:
    ds_stream = load_dataset("ashraq/fashion-product-images-small", split="train", streaming=True)
    categorias_buscadas = {"Apparel": 0}
    for row in ds_stream:
        if row.get("masterCategory") == "Apparel" and row.get("image") is not None:
            imagenes_ropa.append((row["image"], row.get("productDisplayName", "prenda de vestir")))
        if len(imagenes_ropa) >= 4:
            break
    print(f"Descargadas {len(imagenes_ropa)} imágenes reales de prendas desde HF Hub.")
except Exception as exc:
    print(f"Fallback local para imágenes de prueba ({exc}).")
    for nombre, color in [("chaqueta negra", (30, 30, 30)), ("abrigo beige", (210, 190, 160)),
                          ("camisa azul", (50, 90, 160)), ("vestido rojo", (180, 40, 50))]:
        im = Image.new("RGB", (256, 256), (245, 245, 245))
        draw = ImageDraw.Draw(im)
        draw.rectangle([68, 40, 188, 210], fill=color, outline=(10, 10, 10), width=3)
        imagenes_ropa.append((im, nombre))

# Crear 2 imágenes adversariales (fuera de dominio)
img_adv_grafico = Image.new("RGB", (256, 256), (255, 255, 255))
draw_g = ImageDraw.Draw(img_adv_grafico)
draw_g.rectangle([30, 150, 70, 220], fill=(0, 120, 215))
draw_g.rectangle([90, 100, 130, 220], fill=(220, 50, 50))
draw_g.rectangle([150, 60, 190, 220], fill=(40, 170, 80))

rng = np.random.default_rng(SEED)
img_adv_ruido = Image.fromarray(rng.integers(0, 256, (256, 256, 3), dtype=np.uint8))

# Guardar imágenes en notebooks/M4/imagenes_test/ para reproducibilidad
rutas_img = {}
for idx, (im, desc) in enumerate(imagenes_ropa, start=1):
    p = f"{M4_DIR}/imagenes_test/ropa_0{idx}.png"
    im.save(p)
    rutas_img[f"ropa_0{idx}"] = (p, desc)

p_adv1 = f"{M4_DIR}/imagenes_test/adv_grafico.png"
p_adv2 = f"{M4_DIR}/imagenes_test/adv_ruido.png"
img_adv_grafico.save(p_adv1)
img_adv_ruido.save(p_adv2)

casos_imagen = [
    {
        "id": "m4_img_01",
        "tipo": "con_imagen",
        "input": "Quiero un outfit que combine con esta prenda para salir el fin de semana.",
        "image_path": rutas_img["ropa_01"][0],
        "expected_response": "Recomendación de outfits reales del catálogo coherentes con la prenda mostrada en la imagen, citando sus IDs.",
    },
    {
        "id": "m4_img_02",
        "tipo": "con_imagen",
        "input": "Busco algo similar a esto para el trabajo o una reunión formal.",
        "image_path": rutas_img["ropa_02"][0],
        "expected_response": "Recomendación de outfits del catálogo acordes a la prenda de la foto y ocasión formal/trabajo, citando IDs.",
    },
    {
        "id": "m4_img_03",
        "tipo": "con_imagen",
        "input": "Recomiéndame un conjunto parecido a esta imagen.",
        "image_path": rutas_img["ropa_03"][0],
        "expected_response": "Recomendación de outfits del catálogo coherentes con la imagen adjunta, citando sus identificadores.",
    },
    {
        "id": "m4_img_04",
        "tipo": "con_imagen",
        "input": "¿Qué opciones del catálogo encajan con este estilo?",
        "image_path": rutas_img["ropa_04"][0],
        "expected_response": "Recomendación fundamentada en outfits del catálogo alineados al estilo visual de la foto, citando IDs.",
    },
    {
        "id": "m4_adv_01",
        "tipo": "fuera_de_corpus",
        "input": "¿Qué outfit me recomiendas basado en esta imagen?",
        "image_path": p_adv1,
        "expected_response": "No encuentro esa información en el catálogo porque la imagen no corresponde a prendas de vestir o moda.",
    },
    {
        "id": "m4_adv_02",
        "tipo": "fuera_de_corpus",
        "input": "Combina esta prenda por favor.",
        "image_path": p_adv2,
        "expected_response": "No encuentro esa información en el catálogo porque la imagen no muestra ropa ni moda.",
    },
]

# Unir eval set completo (12 texto + 6 imagen = 18 casos)
EVAL_SET_M4 = []
for e in eval_texto_m3:
    copia = dict(e)
    copia["image_path"] = None
    EVAL_SET_M4.append(copia)
EVAL_SET_M4.extend(casos_imagen)

EVAL_M4_OUT = f"{M4_DIR}/eval/eval_set_m4.json"
with open(EVAL_M4_OUT, "w", encoding="utf-8") as f:
    json.dump(EVAL_SET_M4, f, ensure_ascii=False, indent=2)

print(f"Eval set M4 guardado en {EVAL_M4_OUT}: {len(EVAL_SET_M4)} casos totales (12 texto + 6 imagen).")

README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

Descargadas 4 imágenes reales de prendas desde HF Hub.
Eval set M4 guardado en notebooks/M4/eval/eval_set_m4.json: 18 casos totales (12 texto + 6 imagen).


# Evaluación con el Harness de M2 (Tabla Evolutiva + Tabla Delta Visual + Latencias p50/p95)

In [13]:
import gc
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))


JUDGE_MODEL_ID = "google/gemma-2-2b-it"  # Alternativa sin token HF: "microsoft/Phi-3.5-mini-instruct"

_ID_RE = re.compile(r"[BC]_b\d+")
_IDS_VALIDOS = set(doc_ids)

def d1_citacion(eval_set, respuestas, outfits_recuperados):
    scores = []
    for e, r, recup in zip(eval_set, respuestas, outfits_recuperados):
        citados = set(_ID_RE.findall(r or ""))
        reales = {c for c in citados if c in _IDS_VALIDOS}
        if e["tipo"] == "fuera_de_corpus":
            scores.append(1.0 if not citados else 0.0)
        else:
            if not citados:
                scores.append(0.0)
            else:
                recup_set = set(recup or [])
                bien = sum(1 for c in reales if c in recup_set)
                scores.append(bien / len(citados))
    return scores

RUBRICA_QA = (
    "Eres un juez experto evaluando a un personal shopper de moda basado en un catálogo. Te doy la "
    "PREGUNTA, la RESPUESTA ESPERADA y la RESPUESTA DEL SISTEMA. Califica la RESPUESTA DEL SISTEMA "
    "de 1 a 5:\n"
    "5 = recomienda outfits del catálogo coherentes, los cita y no inventa; o activa correctamente "
    "la válvula de escape cuando la pregunta o imagen está fuera del catálogo.\n"
    "4 = buena con un detalle flojo.\n3 = parcial (recupera algo pero se desvía o no cita).\n"
    "2 = pobre o poco fundamentada.\n"
    "1 = inventa prendas/datos, o NO activa la válvula cuando debía.\n"
    'Responde SOLO un JSON: {"puntaje": <1-5>, "motivo": "..."}.'
)

def d2_juez_independiente(eval_set, respuestas):
    print(f"Cargando juez independiente ({JUDGE_MODEL_ID})...")
    judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
    judge_model = AutoModelForCausalLM.from_pretrained(
        JUDGE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map=None,
    ).to(DEVICE)
    judge_model.eval()

    puntajes = []
    with torch.no_grad():
        for e, r in zip(eval_set, respuestas):
            prompt_juez = (
                f"{RUBRICA_QA}\n\n"
                f"PREGUNTA: {e['input']}\n\n"
                f"RESPUESTA ESPERADA: {e['expected_response']}\n\n"
                f"RESPUESTA DEL SISTEMA: {r}\n\nJSON:"
            )
            msgs = [{"role": "user", "content": prompt_juez}]
            inp_txt = judge_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inputs = judge_tok(inp_txt, return_tensors="pt").to(DEVICE)

            out = judge_model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=False,
                pad_token_id=judge_tok.eos_token_id,
            )
            raw = judge_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            obj = parse_json_seguro(raw)
            try:
                p = int(obj.get("puntaje"))
            except Exception:
                p = 3
            puntajes.append(min(5, max(1, p)))

    # Liberar VRAM del juez al terminar la evaluación
    del judge_model, judge_tok
    gc.collect()
    torch.cuda.empty_cache()
    return puntajes

def activo_valvula(resp: str) -> bool:
    return VALVULA.lower() in (resp or "").lower()

def d3_cumplimiento(eval_set, respuestas, d2, umbral: int = 4):
    out = []
    for e, r, p in zip(eval_set, respuestas, d2):
        if e["tipo"] == "fuera_de_corpus":
            out.append(1 if activo_valvula(r) else 0)
        else:
            out.append(1 if (p >= umbral and not activo_valvula(r)) else 0)
    return out

def evaluar_harness_m4(eval_set, nombre: str):
    respuestas, outfits, latencias = [], [], []
    for e in eval_set:
        im = Image.open(e["image_path"]) if e.get("image_path") else None
        res = sistema_multimodal(e["input"], imagen=im)
        respuestas.append(res["respuesta"])
        outfits.append(res["outfits"])
        latencias.append(res["latencia_s"])

    d1 = d1_citacion(eval_set, respuestas, outfits)
    d2 = d2_juez_independiente(eval_set, respuestas)
    d3 = d3_cumplimiento(eval_set, respuestas, d2)

    scorecard = {
        "sistema": nombre,
        "n_casos": len(eval_set),
        "D1_citacion": round(float(np.mean(d1)), 3),
        "D2_juez": round(float(np.mean(d2)), 3),
        "D3_cumplimiento": round(float(np.mean(d3)), 3),
        "latencia_p50_s": round(float(np.percentile(latencias, 50)), 3),
        "latencia_p95_s": round(float(np.percentile(latencias, 95)), 3),
        "vram_max_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2),
    }
    return scorecard, {"respuestas": respuestas, "outfits": outfits, "d1": d1, "d2": d2, "d3": d3, "lat": latencias}

# 1. Evaluación sobre los 12 casos de texto (comparación directa contra M2 y M3)
sc_m4_texto, _ = evaluar_harness_m4(EVAL_SET_M4[:12], "M4 Integrado (12 casos texto)")

# 2. Evaluación sobre los 6 casos con imagen (Delta visual)
sc_m4_imagen, det_img = evaluar_harness_m4(EVAL_SET_M4[12:], "M4 Multimodal (6 casos imagen)")

# 3. Evaluación completa (18 casos: texto + imagen) — Línea base "Antes" para M5
sc_m4_total, det_total = evaluar_harness_m4(EVAL_SET_M4, "M4 Completo (18 casos: texto+imagen)")

df_resultados_m4 = pd.DataFrame([sc_m4_texto, sc_m4_imagen, sc_m4_total])
df_resultados_m4.to_csv(f"{M4_DIR}/resultados/scorecard_m4.csv", index=False)
display(df_resultados_m4)

Cargando juez independiente (google/gemma-2-2b-it)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

... (7 outputs left out)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Cargando juez independiente (google/gemma-2-2b-it)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

,sistema,n_casos,D1_citacion,D2_juez,D3_cumplimiento,latencia_p50_s,latencia_p95_s,vram_max_gb
0,M4 Integrado (12 casos texto),12,0.833,3.917,0.750,9.875,13.317,12.45
1,M4 Multimodal (6 casos imagen),6,0.833,3.833,0.667,15.932,17.776,12.45
2,M4 Completo (18 casos: texto+imagen),18,0.833,3.889,0.722,10.476,16.837,12.45


# Prueba de Invarianza Visual

In [14]:
def ejecutar_prueba_invarianza(ruta_imagen: str, pregunta: str):
    """Comprueba que perturbaciones visuales menores no cambien el diagnóstico de dominio ni los outfits."""
    img_orig = Image.open(ruta_imagen).convert("RGB")
    w, h = img_orig.size
    margen_w, margen_h = int(w * 0.075), int(h * 0.075)

    variantes = {
        "original": img_orig,
        "flip_horizontal": ImageOps.mirror(img_orig),
        "brillo_mas_20": ImageEnhance.Brightness(img_orig).enhance(1.20),
        "recorte_85pct": img_orig.crop((margen_w, margen_h, w - margen_w, h - margen_h)).resize((w, h)),
    }

    resultados_inv = {}
    for nombre_var, im_var in variantes.items():
        res = sistema_multimodal(pregunta, imagen=im_var)
        resultados_inv[nombre_var] = {
            "es_moda": res["info_visual"]["es_moda"],
            "valvula": res["valvula_activada"],
            "outfits": res["outfits"],
            "perfil_m1": res["perfil_m1"],
        }

    base_outfits = set(resultados_inv["original"]["outfits"])
    filas_inv = []
    for nombre_var, datos in resultados_inv.items():
        ov = len(base_outfits & set(datos["outfits"])) / max(1, len(base_outfits))
        filas_inv.append({
            "variante": nombre_var,
            "es_moda_detectado": datos["es_moda"],
            "valvula_activada": datos["valvula"],
            "solapamiento_outfits_vs_orig": round(ov, 2),
            "top_3_outfits": ", ".join(datos["outfits"][:3]),
        })
    return pd.DataFrame(filas_inv)

df_invarianza = ejecutar_prueba_invarianza(
    rutas_img["ropa_01"][0],
    "Quiero un outfit que combine con esta prenda para salir el fin de semana."
)
df_invarianza.to_csv(f"{M4_DIR}/resultados/invarianza_m4.csv", index=False)
display(df_invarianza)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWar

,variante,es_moda_detectado,valvula_activada,solapamiento_outfits_vs_orig,top_3_outfits
0,original,True,False,1.0,"C_b69, C_b6, C_b103"
1,flip_horizontal,True,False,0.6,"C_b69, C_b6, C_b102"
2,brillo_mas_20,True,False,0.8,"C_b6, C_b25, C_b103"
3,recorte_85pct,True,False,0.0,"C_b288, C_b121, C_b152"
